In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql import functions as F
import os

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Patient")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.config("spark.sql.session.timeZone", "UTC")
.master("local[*]")
.getOrCreate())

In [ ]:
silver_base_path = "../../data_lake/silver/silver_Patient/"
gold_base_path = "../../data_lake/gold/dim_patient/"
gold_dimdate = "../../data_lake/gold/dim_date/"
gold_dimtime = "../../data_lake/gold/dim_time/"

In [ ]:
df_patient = spark.read.format("parquet").load(silver_base_path)
df_dimdate = spark.read.format("parquet").load(gold_dimdate)
df_dimtime = spark.read.format("parquet").load(gold_dimtime)

In [ ]:
df_inter = (df_patient.alias("A").join(
    df_dimdate.alias("B"),
    (col("A.birth_date") == col("B.date")),
    "left"
    )
    .join(
    df_dimdate.alias("C"),
    ((col("A.deceased_date_time").cast("date")) == col("C.date")),
    "left"
    )
    .join(
    df_dimtime.alias("D"),
    ((F.hour(col("A.deceased_date_time")) * 3600) + (F.minute(col("A.deceased_date_time")) * 60) + F.second(col("A.deceased_date_time")) == col("D.time_key")),
    "left"
    )
    .select(
    F.conv(F.substring(F.md5("A.patient_id"), 1, 15), 16, 10).cast("bigint").alias("patient_key"),
    col("A.patient_id"),
    col("A.medical_record_number"),
    col("A.social_security_number"),
    col("A.drivers_license_number"),
    col("A.passport_number"),
    col("A.first_name"),
    col("A.middle_name"),
    col("A.family_name").alias("last_name"),
    col("A.prefix"),
    col("A.mothers_maiden_name"),
    col("A.gender"),
    col("A.birth_sex"),
    col("B.date_key").alias("birth_date_key"),
    col("A.birth_city"),
    col("A.birth_state"),
    col("A.birth_country"),
    F.when(col("A.deceased_date_time").isNotNull(), True).otherwise(False).alias("deceased_boolean"),
    col("C.date_key").alias("deceased_date_key"),
    col("D.time_key").alias("deceased_time_key"),
    col("A.race"),
    col("A.ethnicity"),
    col("A.language_display").alias("language"),
    col("A.phone_number"),
    col("A.address_line"),
    col("A.city"),
    col("A.state"),
    col("A.postal_code"),
    col("A.country"),
    col("A.latitude"),
    col("A.longitude"),
    col("A.marital_status_display").alias("marital_status"),
    col("A.is_multiple_birth"),
    F.current_timestamp().alias("gold_timestamp")

))


In [ ]:
df_inter.write.mode("overwrite").format("parquet").save(gold_base_path)